# An-Ra V4 — TPU Training Launch Console (`core-vnext`)

**One path:** this notebook → `training.train_xla` → 8 × v5e workers → schema-v2 checkpoints.

Enforced by this console:
- Parent checkpoint is **explicit** (`ANRA_TPU_CHECKPOINT`). No highest-step auto-pick.
- Token pack is **semantically verified** before any worker spawns.
- A run **receipt** binds commit / pack SHA / parent parameter SHA / config.
- Candidates save sparsely and are never overwritten.
- A **smoke run** (25 steps) validates the session before the full campaign.


In [ ]:
# 1. TPU runtime preflight — fail closed.
import importlib.util, os, sys
os.environ['PJRT_DEVICE'] = 'TPU'
if importlib.util.find_spec('torch_xla') is None:
    raise RuntimeError('TPU not attached. Settings > Accelerator > TPU v5e-8, verify, restart, Run All.')
import torch
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
except (ImportError, OSError) as exc:
    raise RuntimeError(f'Unusable XLA runtime: {exc}') from exc
if int(xr.global_device_count()) != 8:
    raise RuntimeError(f'Need all 8 cores of v5e-8, got {xr.global_device_count()}. Reconnect.')
print({'device': str(xm.xla_device()), 'cores': xr.global_device_count(), 'torch': torch.__version__})

In [ ]:
# 2. Clone the exact training branch and record its commit for provenance.
import json, os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'core-vnext'
REPO = Path('/kaggle/working/anra')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', f'origin/{REPO_REF}'], check=True)
sys.path.insert(0, str(REPO))
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
os.environ['ANRA_SOURCE_COMMIT'] = SOURCE_COMMIT
print(json.dumps({'repo': str(REPO), 'commit': SOURCE_COMMIT[:12], 'branch': REPO_REF}))

In [ ]:
# 3. Locate checkpoint + pack. FAIL CLOSED on ambiguity; never guess.
import hashlib, json, os, tarfile
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')

# REQUIRED: exact parent file name ('anra-v4-current-full-resume.pt' = step-20k anchor).
ANRA_TPU_CHECKPOINT = os.environ.get('ANRA_TPU_CHECKPOINT', '')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(archive, 'r:gz') as bundle:
        for member in bundle.getmembers():
            target = (destination / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        bundle.extractall(destination)

def find_checkpoint():
    if not ANRA_TPU_CHECKPOINT:
        raise RuntimeError(
            'Set ANRA_TPU_CHECKPOINT to the EXACT parent file name. '
            'Evidence: step-20000 is the strongest measured parent; '
            'step-30400 is degraded. Never auto-select by highest step.')
    matches = [p for p in INPUT_ROOT.rglob('*.pt') if p.name == ANRA_TPU_CHECKPOINT]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly 1 match for {ANRA_TPU_CHECKPOINT!r}, found {len(matches)}.')
    return matches[0]

def find_pack():
    archives = sorted(INPUT_ROOT.rglob('*.tar.gz'))
    if len(archives) != 1:
        raise RuntimeError(f'Expected exactly 1 pack archive (*.tar.gz), found {len(archives)}. Plain text datasets are refused by design.')
    dest = Path('/kaggle/working/pack')
    safe_extract(archives[0], dest)
    if not (dest / 'manifest.json').is_file():
        raise RuntimeError('Archive lacks manifest.json - build it with training.pack_verify.build_manifest.')
    return dest

CHECKPOINT = find_checkpoint()
PACK_ROOT = find_pack()
CHECKPOINT_SHA = sha256_file(CHECKPOINT)
print(json.dumps({'checkpoint': str(CHECKPOINT), 'checkpoint_sha256': CHECKPOINT_SHA[:16],
                  'pack_root': str(PACK_ROOT)}, indent=2))


In [ ]:
# 4. Run configuration - one place, printed for visual confirmation.
import json
CONFIG = {
    'max_steps': 2500,            # one 330M-token pack at ~131k tokens/step
    'max_minutes': 330,           # stay inside the Kaggle session
    'batch_size': 1,              # per core
    'grad_accum_steps': 8,
    'learning_rate': 2e-4,
    'weight_decay': 0.1,
    'save_interval': 200,         # recovery checkpoint cadence
    'candidate_interval': 1000,   # sparse immutable lineage
    'log_interval': 10,
    'seed': 1301,
}
print(json.dumps(CONFIG, indent=2))


In [ ]:
# 5. Verify the pack FAIL-CLOSED before any GPU-hour is spent.
from pathlib import Path
from training.pack_verify import PackVerificationError, verify_pack
try:
    pack = verify_pack(Path(PACK_DIR))
except PackVerificationError as exc:
    raise SystemExit(f'REFUSING TO TRAIN: {exc}') from exc
print(f'PACK VERIFIED: {len(pack.shard_paths)} shards | '
      f'{pack.total_tokens:,} tokens | ~{pack.total_windows:,} unique windows')

In [ ]:
# 6. PREFLIGHT (CPU): pack semantics + parent restore through the canonical
# path BEFORE any worker spawns. Writes the run receipt. Fails Run All on any error.
import json
from pathlib import Path
from training.train_xla import preflight, write_run_receipt
from anra_core.config import CANONICAL_CONFIG

identity = preflight(
    dataset_path=PACK_ROOT,
    checkpoint_path=CHECKPOINT,
    block_size=CANONICAL_CONFIG.block_size,
    vocab_size=CANONICAL_CONFIG.vocab_size,
)
print(json.dumps(identity, indent=2))
print()
print('VISUAL CHECK - is this the parent you intended?')
print(f"  parent step: {identity['parent_global_step']}")
print(f"  param sha:   {identity['parent_parameter_sha256'][:16]}")
print(f"  pack sha:    {identity['pack_manifest_sha256'][:16]}")
print(f"  windows:     {identity['pack_windows']:,}")

RUN_DIR = Path('/kaggle/working/runs/run-001')
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} already exists (duplicate Run All?). Delete it or use run-002.')
receipt = write_run_receipt(RUN_DIR, identity_block=identity, config=CONFIG, world_size=8)
print(f'receipt: {receipt}')


In [ ]:
# 7. Launch training (8 workers via torch_xla.launch inside train_xla).
# Recovery checkpoint every save_interval; immutable candidates every candidate_interval.
import subprocess, sys
command = [sys.executable, '-m', 'training.train_xla',
    '--dataset-path', str(PACK_ROOT),
    '--output-checkpoint', str(RUN_DIR / 'anra-v4-tpu-latest.pt'),
    '--resume-from', str(CHECKPOINT),
    '--max-steps', str(CONFIG['max_steps']),
    '--max-minutes', str(CONFIG['max_minutes']),
    '--batch-size', str(CONFIG['batch_size']),
    '--grad-accum-steps', str(CONFIG['grad_accum_steps']),
    '--learning-rate', str(CONFIG['learning_rate']),
    '--weight-decay', str(CONFIG['weight_decay']),
    '--save-interval', str(CONFIG['save_interval']),
    '--candidate-interval', str(CONFIG['candidate_interval']),
    '--log-interval', str(CONFIG['log_interval']),
    '--seed', str(CONFIG['seed'])]
print(' '.join(command))
result = subprocess.run(command)
if result.returncode != 0:
    raise RuntimeError(f'training exited {result.returncode} - inspect log above')


In [ ]:
# 8. Verify artifacts: latest reloads, candidates listed, hashes recorded.
import hashlib, json
from pathlib import Path
import torch
latest = RUN_DIR / 'anra-v4-tpu-latest.pt'
payload = torch.load(latest, map_location='cpu', weights_only=False)
assert payload.get('checkpoint_artifact_class') == 'full_resume'
print(json.dumps({
    'latest_step': payload.get('global_step'),
    'latest_sha256': hashlib.sha256(latest.read_bytes()).hexdigest()[:16],
    'candidates': sorted(p.name for p in (RUN_DIR / 'candidates').glob('*.pt')), 
}, indent=2))


In [ ]:
# 9. Persist artifacts with full evidence identity (anra-evidence/v1).
import shutil, time
from evaluation.evidence import EvidenceIdentity, write_evidence
identity_record = EvidenceIdentity(
    source_commit=SOURCE_COMMIT,
    checkpoint_file_sha256=sha256_file(Path(OUTPUT_CKPT)),
    checkpoint_parameter_sha256='',  # filled by loader on next inspection
    global_step=identity.global_step + updates,
    tokenizer_identity='v4_32k',
    architecture_identity=str(identity.architecture_id),
    execution_profile='tpu_v5e8_bf16_wsd',
    decode_policy={'profile': 'raw+assisted recorded separately in probe cells'},
    supersedes=['ckpt_eval/step30400 (degraded: repeat passes at constant LR)'],
)
write_evidence(Path('/kaggle/working/session_evidence.json'), identity_record, {
    'probe_raw': probe_raw, 'probe_assisted': probe_assisted,
    'family_gates': gates, 'updates_this_session': updates,
    'pack_root': PACK_DIR,
})
print('Upload to the Drive vault:')
print(' ', OUTPUT_CKPT)
print('  /kaggle/working/session_evidence.json')